# 03 — 라벨 재현

**무엇을 확인하는가**

1. 상위 코드의 규칙 그대로 라벨을 다시 만든다
2. 배포 라벨(`Life labels/*.json`)과 **셀 단위로** 맞춰본다
3. 어긋나는 셀을 하나하나 짚는다
4. 도메인 롤업을 낸다
5. 결과를 `experiments/results/nb03_*.json` 에 남긴다 (LOCK 대상)

## 이것이 실패가 아니다

배포 라벨과 우리 계산이 다른 것은 **발견입니다.** 논문 시점과 현재 배포
데이터가 달라 라벨이 갈리는 것은 정상입니다.

**실패는 값이 다른데 어느 셀이 어떻게 다른지 못 짚는 것입니다.**

## 경로가 하나가 아니다

| 서브셋 | 경로 | 재현 |
|---|---|---|
| XJTU | 마지막 하강 구간 **선형 보간** | 가능 |
| Farasis | 외부 Excel, 단위가 **EFC** | **불가** |
| CALB | 외부 Excel 요약표, λ=0.9 | **불가** |
| 나머지 | SOH 가 λ 아래로 내려간 **첫 사이클** | 가능 |

Farasis · CALB 는 대조에서 빠지되 **표에는 남깁니다.** 지우면 "확인했는데
문제없음" 과 구별되지 않습니다.

## 0. 부트스트랩

In [ ]:
import sys
from pathlib import Path

# 노트북에서 저장소 루트를 import 경로에 넣습니다.
REPO = Path.cwd()
while not (REPO / "run.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("저장소 루트:", REPO)

from verify import load_config, write_json
from verify import labels as labels_mod

config = load_config()
EXTRACT = Path(config["EXTRACT_DIR"])
LABELS_DIR = EXTRACT / "Life labels"    # 폴더 이름에 공백이 있습니다
RESULTS = REPO / "experiments" / "results"

print("라벨 폴더:", LABELS_DIR, "[있음]" if LABELS_DIR.exists() else "[없음]")
if LABELS_DIR.exists():
    for path in sorted(LABELS_DIR.iterdir()):
        print("   ", path.name)

## 1. 경로 판정

계산 전에 **먼저** 어느 경로인지 정합니다.

In [ ]:
subset_dirs = sorted(
    p.name for p in EXTRACT.iterdir()
    if p.is_dir() and p.name not in ("Life labels", "READMEs")
)
for name in subset_dirs:
    print(f"  {name:14} {labels_mod.route_of(name):8} "
          f"λ={labels_mod.lambda_of(name)}  도메인={labels_mod.domain_of(name)}  "
          f"라벨파일={labels_mod.label_json_name(name)}")

## 2. 재현 + 대조

`limit` 을 걸지 마십시오. 대조표는 전 셀이어야 뜻이 있습니다.

In [ ]:
all_rows = []
for name in subset_dirs:
    rows = labels_mod.label_subset(name, EXTRACT)
    try:
        distributed = labels_mod.load_distributed_labels(LABELS_DIR, name)
    except FileNotFoundError as error:
        print(f"  {name:14} 배포 라벨 없음 — {error}")
        distributed = {}
    rows = labels_mod.compare(rows, distributed)
    all_rows.extend(rows)

    counts = {}
    for row in rows:
        counts[row["match"]] = counts.get(row["match"], 0) + 1
    summary = "  ".join(f"{k} {v}" for k, v in sorted(counts.items()))
    print(f"  {name:14} {len(rows):4} cells   {summary}")

print(f"\n총 {len(all_rows)} cells")

## 3. 불일치 셀 — 하나하나

**여기가 이 저장소의 핵심입니다.** 값이 다르다는 것으로 끝내지 말고 어느
셀이 어느 경로에서 얼마나 어긋나는지 봅니다.

In [ ]:
mismatch = [row for row in all_rows if row["match"] == "불일치"]
print(f"불일치 {len(mismatch)}개\n")

for row in mismatch[:40]:
    print(f"  {row['subset']:10} {row['cell']:32} "
          f"우리 {str(row['label']):>7}  배포 {str(row['theirs']):>7}  "
          f"차 {str(row.get('delta')):>7}  [{row['status']}]")
if len(mismatch) > 40:
    print(f"  ... 외 {len(mismatch) - 40}개 (nb03_mismatch.json 에 전부 있습니다)")

print("\n경로별 불일치:")
by_route = {}
for row in mismatch:
    by_route[row["status"]] = by_route.get(row["status"], 0) + 1
for status, count in sorted(by_route.items(), key=lambda kv: -kv[1]):
    print(f"  {status:32} {count}")

## 4. 한쪽에만 있는 셀

- `우리만있음` — 우리는 라벨을 만들었는데 배포 파일에 없습니다. 배포 시점에
  폐기됐거나 다른 규칙이었다는 뜻입니다.
- `배포만있음` — 배포에는 있는데 우리는 못 만들었습니다. **폐기 임계에서
  걸러졌는지 먼저 보십시오** (02 의 폐기 개수와 맞춰봅니다).

In [ ]:
for kind in ("우리만있음", "배포만있음"):
    rows = [row for row in all_rows if row["match"] == kind]
    print(f"\n{kind}: {len(rows)}개")
    for row in rows[:20]:
        print(f"  {row['subset']:10} {row['cell']:32} "
              f"우리 {str(row['label']):>7}  배포 {str(row['theirs']):>7}  [{row['status']}]")
    if len(rows) > 20:
        print(f"  ... 외 {len(rows) - 20}개")

## 5. 라벨없음(비유한) — XJTU 의 NaN 은 정상

상위 코드가 `np.isfinite` 로 유효 개수를 따로 세고 `80% labels: {n}` 을
출력합니다. **오류가 아닙니다.** `LAB-011` 은 이 NaN 이 배포 라벨 파일에도
들어 있는지를 묻습니다 — 아래에서 바로 확인됩니다.

In [ ]:
nolabel = [row for row in all_rows if row["match"] == "대조제외(라벨없음)"]
print(f"라벨없음(비유한) {len(nolabel)}개")
for row in nolabel[:20]:
    print(f"  {row['subset']:10} {row['cell']:32} 배포 {row['theirs']!r}")

# LAB-011: 배포 XJTU 라벨 파일에 NaN 이 들어 있는가
import json
xjtu_path = LABELS_DIR / "XJTU_labels.json"
if xjtu_path.exists():
    raw = xjtu_path.read_text(encoding="utf-8")
    print(f"\nXJTU_labels.json 안의 NaN 표기: {raw.count('NaN')}개")
    print("→ LAB-011 의 code 슬롯을 이 결과로 채우십시오.")

## 6. 재현불가 — 표에 남긴다

Farasis · CALB 는 외부 Excel 이 배포되지 않아 **구조적으로** 확인 불가입니다.
`구조적불가` 는 실패가 아니라 종결된 판정입니다.

In [ ]:
unrep = [row for row in all_rows if row["match"] == "대조제외(재현불가)"]
by_subset = {}
for row in unrep:
    by_subset[row["subset"]] = by_subset.get(row["subset"], 0) + 1
for subset, count in sorted(by_subset.items()):
    print(f"  {subset:10} {count:4} cells   재현불가(외부파일)")
print("\n지우지 마십시오. 표에 남아 있어야 '확인했는데 문제없음' 과 구별됩니다.")

## 7. 도메인 롤업

In [ ]:
rollup = labels_mod.rollup(all_rows)
header = f"{'도메인':10} {'총':>6} {'라벨':>6} {'절단':>6} {'폐기':>6} {'외삽':>6} {'재현불가':>8} {'절단비':>8}"
print(header)
print("-" * len(header))
for row in rollup:
    print(f"{row['domain']:10} {row['cells']:6} {row['labeled']:6} "
          f"{row['censored']:6} {row['abandoned']:6} {row['extrapolated']:6} "
          f"{row['unreproducible']:8} {str(row['censored_ratio']):>8}")

## 8. LOCK 대상 산출물 저장

`normalized=True` 로 씁니다. **렌더링된 마크다운이 아니라 정규화 JSON 을
해싱** 하기 때문입니다 — 공백 하나로 어긋나지 않게.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)

write_json(RESULTS / "nb03_cells.json", all_rows, normalized=True)
write_json(RESULTS / "nb03_rollup.json", rollup, normalized=True)
write_json(RESULTS / "nb03_mismatch.json", mismatch, normalized=True)
write_json(RESULTS / "nb03_nolabel.json", nolabel, normalized=True)

for name in ("nb03_cells", "nb03_rollup", "nb03_mismatch", "nb03_nolabel"):
    print("저장:", RESULTS / f"{name}.json")

print("\n이제 python run.py lock-init 이 nb03 항목들을 채울 수 있습니다.")

## 9. 다음

1. 불일치 셀 목록을 보고 **어느 규칙에서 갈리는지** 를 정하십시오.
   경로별 집계(3절)가 출발점입니다.
2. 갈리는 규칙마다 `findings/registry.yaml` 의 해당 `LAB-` 레코드에 관찰을
   적으십시오. `code` 슬롯은 이미 차 있으니 `note` 에 셀 목록을 겁니다.
3. `python run.py claims` 로 문서를 다시 만드십시오.

**원인을 지어내지 마십시오.** 못 찾으면 `조사했으나불명` 이고, 확인 경로가
없으면 `구조적불가` 입니다. 둘 다 정상적인 결말입니다.